# Superheterodyne Receiver

This notebook turns the receiver-chain concepts into a concrete signal path: RF filtering, mixing to an intermediate frequency, IF selection, and demodulation.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## Why Convert to an IF?

Mixing shifts a desired channel from its RF location down to a fixed intermediate frequency where narrow, stable filters are easier to build and reason about.

In [ ]:
fs = 192_000
t = np.arange(0, 0.03, 1 / fs)
rf_stations = [
    (40_000, 0.9, 800),
    (55_000, 0.6, 1400),
    (72_000, 0.5, 2200),
]
rf_band = np.zeros_like(t)
for carrier, gain, audio_freq in rf_stations:
    rf_band += gain * am_modulate(np.cos(2 * np.pi * audio_freq * t), carrier_freq=carrier, fs=fs, mod_index=0.7)

fig, ax = plt.subplots(figsize=(10, 3.5))
plot_spectrum(rf_band, fs=fs, ax=ax, title="Three RF Channels in the Front End")
ax.set_xlim(0, 90_000)
ax.set_ylim(-110, 5)
plt.tight_layout()


In [ ]:
audio_out = audio_output_widget()
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
IF_FREQ = 12_000

def update_tuning(lo_freq=52_000):
    mixer_lo = np.cos(2 * np.pi * lo_freq * t)
    mixed = rf_band * mixer_lo
    sos = signal.butter(5, [IF_FREQ - 4000, IF_FREQ + 4000], btype="band", fs=fs, output="sos")
    if_signal = signal.sosfilt(sos, mixed)
    demod = am_demodulate(if_signal * np.cos(2 * np.pi * IF_FREQ * t), fs=fs, audio_cutoff=4000)

    for ax in axes:
        ax.clear()
    plot_spectrum(rf_band, fs=fs, ax=axes[0], title="RF Band")
    axes[0].axvline(lo_freq, color="tab:red", linestyle="--", label="LO")
    axes[0].legend()
    plot_spectrum(mixed, fs=fs, ax=axes[1], title="Mixer Output")
    plot_spectrum(if_signal, fs=fs, ax=axes[2], title="Filtered IF")
    axes[0].set_xlim(0, 90_000)
    axes[1].set_xlim(0, 90_000)
    axes[2].set_xlim(0, 30_000)
    for ax in axes:
        ax.set_ylim(-110, 5)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, resample_signal(demod, fs, 44_100), rate=44_100)

controls = widgets.interactive(
    update_tuning,
    lo_freq=float_slider(min_value=44_000, max_value=84_000, step=500, value=52_000, description="LO Hz"),
)
display(controls, audio_out)


## Key Takeaway

The receiver does not directly "listen" at RF and recover audio in one step. It shifts energy through stages, and each stage exists to make selection and demodulation easier.